# Feature groups, model settings and seasonal models

Reviewer point A7 asks to compare the load, calendar and weather feature groups on chronological folds, and to add
features only if they improve validation performance. Reviewer point E2 asks about the weather weighting. Reviewer A
also suggests comparing seasonal models with the single model, tracked in MLflow.

The logic lives in `winterload.forecast.evaluation.feature_study` and `winterload.forecast.evaluation.season_report`, which have unit tests.
This notebook reads the feature store, writes to the result store and logs every comparison to MLflow.

It never reads the test year: `history` stops at `VALIDATION_END`.

In [1]:
import winterload.config  # Loads .env before anything imports fsspec.

import dataclasses

import pandas as pd
from winterload.forecast.evaluation.feature_study import FOLD_COUNT, FOLD_HOURS, feature_sets, out_of_fold_forecast, \
    summarize_folds
from winterload.forecast.evaluation.season_report import compare_seasonal_models, report_by_holiday
from winterload.forecast.evaluation.splits import TRAIN_START, VALIDATION_END
from winterload.forecast.features.dataset import EQUAL_CITY_WEIGHTS, POPULATION_CITY_WEIGHTS
from winterload.forecast.features.holidays_ch import NATIONAL_HOLIDAY_SHARE
from winterload.forecast.model.baseline import seasonal_naive_forecast
from winterload.forecast.model.preliminary import EARLY_STOPPING_SETTINGS, PRELIMINARY_SETTINGS
from winterload.repository.factory import champion_repository, result_repository
from winterload.repository.results import FEATURE_GROUPS, HOLIDAY_ERRORS, MODEL_SETTINGS, ORDER_NOISE, \
    SEASONAL_MODELS, WEATHER_WEIGHTS
from winterload.repository.training_frame import load_dataframe
from winterload.storage import get_storage

RUN_AT = pd.Timestamp.now(tz="UTC").floor("s")

storage = get_storage()
registry = champion_repository()
results = result_repository()
history = load_dataframe(storage, TRAIN_START, VALIDATION_END)
print(f"{len(history)} rows from {history['event_timestamp'].min()} to {history['event_timestamp'].max()}")


def save(name, rows):
    frame = pd.DataFrame(rows)
    results.write(name, frame.assign(run_at=RUN_AT))
    return frame


def scored_positions(scored, frame):
    # Where the scored rows sit in `frame`, matched on their hour.
    return frame["event_timestamp"].searchsorted(scored["event_timestamp"])


def summary_row(summary):
    return {"name": summary.name, "mean_mape": summary.mean_mape, "mean_mae": summary.mean_mae,
            "fold_mapes": " ".join(f"{value:.4f}" for value in summary.fold_mapes)}


def track(study, summary, params=None):
    with registry.run(f"{study}: {summary.name}"):
        registry.log(
            params={"study": study, "rows": len(history), "fold_count": FOLD_COUNT, "fold_hours": FOLD_HOURS}
                   | (params or {}),
            metrics={"mean_mape": summary.mean_mape, "mean_mae": summary.mean_mae}
                    | {f"fold_{i}_mape": value for i, value in enumerate(summary.fold_mapes)},
        )

/Users/bjorn/code/ai/26H.MLOPS-winterload/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


13201 rows from 2024-03-15 23:00:00+00:00 to 2025-09-22 23:00:00+00:00


## Feature groups

Each row trains on one combination of groups. The baseline row scores the seasonal-naive forecast on the same folds.

Columns: `name` is the combination of feature groups. `fold_mapes` holds the MAPE of each of the six 8-week validation
folds, oldest first, and `mean_mape` and `mean_mae` average the folds, each fold counting the same. MAPE is a fraction (0.05 = 5%), MAE is in MW.

In [2]:
scored_sets = {name: out_of_fold_forecast(history, columns) for name, columns in feature_sets().items()}
summaries = [summarize_folds(name, scored) for name, scored in scored_sets.items()]
all_features = next(iter(scored_sets.values()))
summaries.append(summarize_folds("baseline", all_features.assign(forecast=seasonal_naive_forecast(all_features))))
for summary in summaries:
    track("feature_groups", summary)
save(FEATURE_GROUPS, [summary_row(summary) for summary in summaries]).sort_values("mean_mape")

🏃 View run feature_groups: load at: http://localhost:5001/#/experiments/1/runs/c01561c9c28e46e6adc5990d8046464b
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run feature_groups: calendar at: http://localhost:5001/#/experiments/1/runs/acdaed8c0e9d41bf92a3dddbf259685c
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run feature_groups: weather at: http://localhost:5001/#/experiments/1/runs/60a1fe66c6b947c7b797a724ce5f4192
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run feature_groups: load+calendar at: http://localhost:5001/#/experiments/1/runs/ac8ad78f56c24a8e9479baf264965be8
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run feature_groups: load+weather at: http://localhost:5001/#/experiments/1/runs/06dd12d8c55e4bccb16169eb859e2338
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run feature_groups: calendar+weather at: http://localhost:5001/#/experiments/1/runs/0b243e87386748c39186d1eb1

,name,mean_mape,mean_mae,fold_mapes
6,load+calendar+weather,0.067497,466.339192,0.1054 0.0895 0.0387 0.0540 0.0643 0.0532
3,load+calendar,0.077261,541.039835,0.1192 0.1122 0.0511 0.0626 0.0638 0.0547
0,load,0.077793,535.474191,0.1101 0.0992 0.0601 0.0695 0.0680 0.0598
4,load+weather,0.079344,551.031740,0.1077 0.1181 0.0572 0.0654 0.0695 0.0581
7,baseline,0.082514,559.095230,0.1094 0.1045 0.0713 0.0758 0.0713 0.0628
5,calendar+weather,0.091398,648.055772,0.1370 0.1223 0.0474 0.0605 0.0875 0.0937
2,weather,0.102594,725.005728,0.1348 0.1229 0.0772 0.0961 0.0979 0.0867
1,calendar,0.103017,737.026159,0.1433 0.1589 0.0677 0.0755 0.0801 0.0927


## Holiday features

The proposal promises to evaluate holidays separately, and reviewer point A7 says to add features only if they improve
validation performance. Each variant trains on all other features. `share` is the model's `holiday_share_ch`, and `flag`
replaces it with `is_holiday_ch`. `national as Sunday` drops both and sets the calendar fields of national holidays to a
Sunday's. See `winterload.forecast.features.holidays_ch` for the definitions.

Each variant has three rows, one per segment. `mean_mape_all_folds` and `fold_mapes` cover all validation hours, so they
repeat within a variant. `segment` splits those hours into national holidays (a share of at least 50%), regional
holidays (a share above 0) and other days, and `hours` and `months` say what each segment rests on. `model_*` and
`baseline_*` score the segment's hours for the model and the seasonal-naive baseline. MAPE is a fraction (0.05 = 5%), MAE is in MW.

In [3]:
share = history["holiday_share_ch"]
national = share >= NATIONAL_HOLIDAY_SHARE
without_share = history.drop(columns="holiday_share_ch")
variants = {
    "no holiday feature": without_share,
    "share": history,
    "flag": without_share.assign(is_holiday_ch=national.astype(int)),
    "national as Sunday": without_share.assign(day_of_week=history["day_of_week"].where(~national, 6),
                                               is_weekend=history["is_weekend"].where(~national, 1)),
}
holiday_rows = []
for name, frame in variants.items():
    scored = out_of_fold_forecast(frame)
    summary = summarize_folds(name, scored)
    track("holiday_features", summary)
    # Every variant reports on the same holiday hours, so the segments come from the real share.
    for result in report_by_holiday(scored.assign(holiday_share_ch=share.iloc[scored_positions(scored, history)].to_numpy())):
        holiday_rows.append({"variant": name, "mean_mape_all_folds": summary.mean_mape,
                             "fold_mapes": summary_row(summary)["fold_mapes"],
                             **dataclasses.asdict(result) | {"months": " ".join(result.months)}})
save(HOLIDAY_ERRORS, holiday_rows)

🏃 View run holiday_features: no holiday feature at: http://localhost:5001/#/experiments/1/runs/0b04e34533524495b3898b05318d562d
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run holiday_features: share at: http://localhost:5001/#/experiments/1/runs/3d38d74ba0ac45a38217a5a7bed8428c
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run holiday_features: flag at: http://localhost:5001/#/experiments/1/runs/6e12b179f130465da784c456a32cd4c3
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run holiday_features: national as Sunday at: http://localhost:5001/#/experiments/1/runs/d9aec4a7fc57489e9e37e129e6bee79f
🧪 View experiment at: http://localhost:5001/#/experiments/1


,variant,mean_mape_all_folds,fold_mapes,segment,hours,months,model_mape,model_mae,baseline_mape,baseline_mae
0,no holiday feature,0.068108,0.1051 0.0891 0.0390 0.0562 0.0640 0.0553,national holiday,192,2024-12 2025-01 2025-04 2025-05 2025-06 2025-08,0.124514,780.463306,0.120771,754.531667
1,no holiday feature,0.068108,0.1051 0.0891 0.0390 0.0562 0.0640 0.0553,regional holiday,360,2024-11 2024-12 2025-01 2025-03 2025-04 2025-0...,0.074056,516.683653,0.074397,509.460833
2,no holiday feature,0.068108,0.1051 0.0891 0.0390 0.0562 0.0640 0.0553,other day,7512,2024-10 2024-11 2024-12 2025-01 2025-02 2025-0...,0.066382,458.876103,0.081925,556.478695
3,share,0.067497,0.1054 0.0895 0.0387 0.0540 0.0643 0.0532,national holiday,192,2024-12 2025-01 2025-04 2025-05 2025-06 2025-08,0.114872,764.479633,0.120771,754.531667
4,share,0.067497,0.1054 0.0895 0.0387 0.0540 0.0643 0.0532,regional holiday,360,2024-11 2024-12 2025-01 2025-03 2025-04 2025-0...,0.072310,505.640012,0.074397,509.460833
5,share,0.067497,0.1054 0.0895 0.0387 0.0540 0.0643 0.0532,other day,7512,2024-10 2024-11 2024-12 2025-01 2025-02 2025-0...,0.066055,456.835563,0.081925,556.478695
6,flag,0.068181,0.1069 0.0904 0.0401 0.0549 0.0635 0.0533,national holiday,192,2024-12 2025-01 2025-04 2025-05 2025-06 2025-08,0.120841,795.739726,0.120771,754.531667
7,flag,0.068181,0.1069 0.0904 0.0401 0.0549 0.0635 0.0533,regional holiday,360,2024-11 2024-12 2025-01 2025-03 2025-04 2025-0...,0.073954,518.229309,0.074397,509.460833
8,flag,0.068181,0.1069 0.0904 0.0401 0.0549 0.0635 0.0533,other day,7512,2024-10 2024-11 2024-12 2025-01 2025-02 2025-0...,0.066559,460.729330,0.081925,556.478695
9,national as Sunday,0.068095,0.1076 0.0860 0.0390 0.0558 0.0640 0.0562,national holiday,192,2024-12 2025-01 2025-04 2025-05 2025-06 2025-08,0.103763,667.002141,0.120771,754.531667


## Weather weighting

Reviewer point E2. Both variants train on all features and differ only in how the four cities combine.

The columns match the feature-group table.

In [4]:
weight_summaries = []
for name, weights in [("equal weights", EQUAL_CITY_WEIGHTS), ("population weights", POPULATION_CITY_WEIGHTS)]:
    frame = load_dataframe(storage, TRAIN_START, VALIDATION_END, city_weights=weights)
    summary = summarize_folds(name, out_of_fold_forecast(frame))
    weight_summaries.append(summary)
    track("weather_weights", summary, params={f"weight_{city}": weight for city, weight in weights.items()})
save(WEATHER_WEIGHTS, [summary_row(summary) for summary in weight_summaries])

🏃 View run weather_weights: equal weights at: http://localhost:5001/#/experiments/1/runs/0f00fde988c94080b484334f21d168e5
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run weather_weights: population weights at: http://localhost:5001/#/experiments/1/runs/94165759c8e249dd9a9a513546d0d8f7
🧪 View experiment at: http://localhost:5001/#/experiments/1


,name,mean_mape,mean_mae,fold_mapes
0,equal weights,0.067497,466.339192,0.1054 0.0895 0.0387 0.0540 0.0643 0.0532
1,population weights,0.068435,473.977638,0.1085 0.0917 0.0396 0.0533 0.0641 0.0534


## Model settings

The preliminary model keeps LightGBM's defaults, because the B1/A4 target measures tuning against it. This cell
shows what early stopping would add. Early stopping watches the newest 10% of each fold's training rows.

The columns match the feature-group table, plus the LightGBM settings of each variant. `early_stopping_rounds` is empty
for the defaults, which always train 100 trees.

In [5]:
setting_rows = []
for name, settings in [("defaults, 100 trees", PRELIMINARY_SETTINGS),
                       ("early stopping, up to 2000 trees", EARLY_STOPPING_SETTINGS)]:
    summary = summarize_folds(name, out_of_fold_forecast(history, settings=settings))
    track("model_settings", summary, params=dataclasses.asdict(settings))
    setting_rows.append(summary_row(summary) | dataclasses.asdict(settings))
save(MODEL_SETTINGS, setting_rows)

🏃 View run model_settings: defaults, 100 trees at: http://localhost:5001/#/experiments/1/runs/c32afc05af3c49be91cfa3c18c1582c1
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run model_settings: early stopping, up to 2000 trees at: http://localhost:5001/#/experiments/1/runs/155071a04d734412b54602f11da46bc6
🧪 View experiment at: http://localhost:5001/#/experiments/1


,name,mean_mape,mean_mae,fold_mapes,n_estimators,learning_rate,num_leaves,max_depth,min_child_samples,early_stopping_rounds
0,"defaults, 100 trees",0.067497,466.339192,0.1054 0.0895 0.0387 0.0540 0.0643 0.0532,100,0.10,31,-1,20,NaN
1,"early stopping, up to 2000 trees",0.072420,507.448158,0.1186 0.0930 0.0452 0.0564 0.0665 0.0549,2000,0.05,31,-1,20,50.0


## One model per season

Reviewer A suggests this check if seasonal weaknesses persist. Both variants score the same months, each with a model
fit on earlier rows only. The history outside the test year holds one winter, so the winter row rests on few months.

Columns: `hours` and `months` give the scored rows of each season, the same for both variants. `single_*` is the one
model for the whole year, `seasonal_*` a model trained only on earlier rows of the same season, and `baseline_mape` the
seasonal-naive baseline on the same hours. The seasonal model needs 8 weeks of its season first, so the first months
of winter and summer drop out and these counts sit below the replay notebook's. MAPE is a fraction (0.05 = 5%), MAE is in MW.

In [6]:
comparisons = compare_seasonal_models(history)
for comparison in comparisons:
    with registry.run(f"seasonal_models: {comparison.season}"):
        registry.log(
            params={"study": "seasonal_models", "season": comparison.season, "hours": comparison.hours,
                    "months": " ".join(comparison.months)},
            metrics={"single_mape": comparison.single_mape, "seasonal_mape": comparison.seasonal_mape,
                     "single_mae": comparison.single_mae, "seasonal_mae": comparison.seasonal_mae,
                     "baseline_mape": comparison.baseline_mape},
        )
save(SEASONAL_MODELS, [dataclasses.asdict(comparison) | {"months": " ".join(comparison.months)}
                             for comparison in comparisons])

🏃 View run seasonal_models: winter at: http://localhost:5001/#/experiments/1/runs/b2bb42aeb24d4b66a0efc485638189aa
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run seasonal_models: transition at: http://localhost:5001/#/experiments/1/runs/f2252e3942674ae895f04550070de67f
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run seasonal_models: summer at: http://localhost:5001/#/experiments/1/runs/e731f04a72f34842b0a9f70d6eb9f94f
🧪 View experiment at: http://localhost:5001/#/experiments/1


,season,hours,months,single_mape,single_mae,seasonal_mape,seasonal_mae,baseline_mape
0,winter,1416,2025-01 2025-02,0.057249,462.484634,0.055831,449.419370,0.078188
1,transition,4202,2024-09 2024-10 2025-03 2025-04 2025-05 2025-09,0.056613,363.109962,0.063278,406.628560,0.075847
2,summer,2808,2024-08 2025-06 2025-07 2025-08,0.067252,422.726362,0.072612,458.386266,0.074773


## How much does row order move these numbers?

The store returns the four cities of one hour in no fixed order, and the sum of four floats depends on their order in the last bits. LightGBM bins on exact values, so that can change a model. `WeatherRepository` sorts the rows, which pins one reproducible result. This cell shuffles the order to show the spread around it. A difference smaller than this spread says nothing.

The output describes `mean_mape` over the 8 shuffled orders: `std`, `min` and `max` give the spread.

In [7]:
import datetime as dt

from winterload.forecast.features.dataset import LAG_HOURS, aggregate_weather, assemble_training_frame
from winterload.repository.load import LoadRepository
from winterload.repository.weather import WeatherRepository

load = LoadRepository(storage).hourly(TRAIN_START - dt.timedelta(hours=max(LAG_HOURS)), VALIDATION_END)
weather = WeatherRepository(storage).forecast(TRAIN_START, VALIDATION_END)
noise_rows = []
for seed in range(8):
    shuffled = weather.sample(frac=1, random_state=seed).sort_values("event_timestamp", kind="stable").reset_index(drop=True)
    frame = assemble_training_frame(load, aggregate_weather(shuffled), TRAIN_START)
    noise_rows.append({"seed": seed, "mean_mape": summarize_folds("shuffled", out_of_fold_forecast(frame)).mean_mape})
save(ORDER_NOISE, noise_rows)["mean_mape"].describe()

count    8.000000
mean     0.067741
std      0.000112
min      0.067577
25%      0.067649
50%      0.067759
75%      0.067810
max      0.067905
Name: mean_mape, dtype: float64